# Estadísticas zonales en Python — `reduceRegions` y la salida a pandas

**Día 2 · Curso GEE 2026**

Este cuaderno es **el mismo análisis de `07_srtm_munis.js`, traducido**. La idea no es
aprender algo nuevo, es ver que Earth Engine es la misma herramienta con otra piel: las
llamadas son las mismas, con la misma semántica de cliente y servidor, y lo único que
cambia de verdad es **a dónde cae el resultado**.

Y ahí está el motivo para pasarse a Python: en JavaScript el resultado se queda en la
consola del Code Editor y para hacer algo con él hay que exportarlo. Aquí cae en un
`DataFrame` de pandas, que es donde el análisis se vuelve cómodo: ordenar, agrupar,
cruzar con una tabla propia, graficar y guardar.

**Lo que se practica**

- `reduceRegion` contra `reduceRegions`, y por qué el segundo escala
- reductores **combinados** y **agrupados**: varias estadísticas en una sola pasada
- `scale` siempre explícito, porque manda sobre el resultado
- `ee_to_df` para caer en pandas, y de ahí a CSV y a una figura
- los tres errores de memoria y tiempo, y qué palanca tocar en cada uno

## 1. Puesta en marcha

En **Colab** no hay que instalar nada: `ee` y `geemap` ya vienen en el entorno. La primera
vez de cada sesión hay que autenticarse, y en `Initialize` va **tu** proyecto de Google
Cloud (el que se creó el día 1), no el mío.

> Si trabajas en local (VS Code, Antigravity, Jupyter), descomenta el `pip install`.

In [ ]:
# !pip install geemap -q

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-digdgeografo')   # <-- cambia esto por tu proyecto

print('geemap', geemap.__version__)

## 2. Los datos: los mismos del script de JavaScript

`full` es la imagen de tres bandas (elevación, pendiente, aspecto) y `andalucia` la capa
de municipios. Exactamente lo de `07_srtm_munis.js`, línea por línea.

In [ ]:
elevation = ee.Image('CGIAR/SRTM90_V4').select('elevation')
slope = ee.Terrain.slope(elevation)
aspect = ee.Terrain.aspect(elevation)

full = ee.Image.cat([elevation, slope, aspect])

andalucia = ee.FeatureCollection('users/digdgeografo/curso_GEE/Andalucia')

print('Municipios:', andalucia.size().getInfo())
print('Bandas:', full.bandNames().getInfo())

Un mapa interactivo para no trabajar a ciegas. `geemap.Map` es el equivalente del mapa del
Code Editor, y `add_layer` toma los mismos diccionarios de visualización.

In [ ]:
Map = geemap.Map(center=[37.2, -5.0], zoom=7)

Map.add_layer(
    elevation,
    {'min': 0, 'max': 3000,
     'palette': ['green', 'yellow', 'orange', 'brown', 'white']},
    'Elevación (m)',
)
Map.add_layer(andalucia.style(color='black', fillColor='00000000', width=1),
              {}, 'Municipios')
Map

## 3. `reduceRegion`: una geometría, un diccionario

`reduceRegion` resume la imagen sobre **una sola** geometría y devuelve un diccionario.
Es la versión que todo el mundo aprende primero, y la que no escala.

Fíjate en `scale=90`: es la resolución del SRTM. **`scale` no es un ajuste de calidad, es
parte del resultado**: la media de la pendiente a 90 m y a 500 m son dos números
distintos, los dos correctos, que responden a dos preguntas distintas. Si no lo pones,
Earth Engine elige por ti y el número deja de ser reproducible.

In [ ]:
almonte = andalucia.filter(ee.Filter.eq('nombre', 'Almonte'))

stats_almonte = full.select(['elevation', 'slope']).reduceRegion(
    reducer=ee.Reducer.median(),
    geometry=almonte.geometry(),
    scale=90,            # explícito SIEMPRE
    maxPixels=1e9,
)

# .getInfo() trae el resultado del servidor al cliente: una sola vez, al final.
stats_almonte.getInfo()

## 4. `reduceRegions`: muchas geometrías, una tabla

`reduceRegions` aplica el reductor a **cada** *feature* de la colección y devuelve otra
`FeatureCollection` con las propiedades originales más los resultados.

La diferencia no es de comodidad, es de arquitectura: un bucle con `reduceRegion` lanza
una petición por polígono y las paga todas (y se come el
`Too many concurrent aggregations` en cuanto son muchos). `reduceRegions` es **una**
petición que el servidor paraleliza por su cuenta.

> La regla: si hay más de una geometría, `reduceRegions`. Sin excepciones.

In [ ]:
munis = andalucia.filter(
    ee.Filter.inList('nombre', ['Almonte', 'Monachil', 'Cazorla', 'Aroche', 'Níjar'])
)

zonales = full.reduceRegions(
    collection=munis.select(['nombre']),   # quedarse solo con lo que hace falta
    reducer=ee.Reducer.mean(),
    scale=90,
)

print(zonales.first().toDictionary().getInfo())

## 5. Reductores combinados: todas las estadísticas en una pasada

`combine` encadena reductores para que la imagen se lea **una sola vez**. Dos pasadas con
dos reductores cuestan el doble que una pasada con los dos combinados, y el resultado es
idéntico.

`sharedInputs=True` es lo que dice «los dos reductores se aplican a las mismas bandas».
Sin eso, Earth Engine espera una banda por reductor y el error que da no ayuda nada.

In [ ]:
estadisticas = (
    ee.Reducer.mean()
    .combine(ee.Reducer.stdDev(), sharedInputs=True)
    .combine(ee.Reducer.minMax(), sharedInputs=True)
    .combine(ee.Reducer.percentile([10, 50, 90]), sharedInputs=True)
    .combine(ee.Reducer.count(), sharedInputs=True)
)

zonales_full = full.select(['elevation', 'slope']).reduceRegions(
    collection=munis.select(['nombre']),
    reducer=estadisticas,
    scale=90,
)

print(zonales_full.first().toDictionary().getInfo())

Ese `count` del final no es decorativo: es **el número de píxeles que entraron en cada
media**. Un municipio con 40 píxeles y otro con 400.000 aparecen igual de serios en la
tabla, y no lo son. Llévate siempre el recuento.

### Reductores agrupados: el añadido que no tiene JavaScript fácil

`group` parte el resultado por los valores de una banda. Aquí: la elevación media **por
clase de usos del suelo**, en una sola llamada. Es el equivalente de un `GROUP BY`.

In [ ]:
usos = (ee.ImageCollection('COPERNICUS/Landcover/100m/Proba-V-C3/Global')
        .first()
        .select('discrete_classification')
        .rename('clase'))

por_clase = elevation.addBands(usos).reduceRegion(
    reducer=ee.Reducer.mean().group(groupField=1, groupName='clase'),
    geometry=almonte.geometry(),
    scale=100,
    maxPixels=1e9,
)

por_clase.getInfo()

## 6. La caída a pandas

Hasta aquí todo pasaba en los servidores de Google. `ee_to_df` trae la tabla al cuaderno y
desde ese momento es pandas normal.

**Esta línea es la frontera cliente/servidor.** Todo lo de antes era perezoso y no había
corrido nada; aquí se ejecuta de golpe. Si va a tardar, tarda aquí — y si revienta,
revienta aquí, que es la razón por la que conviene que esta llamada esté sola en su celda.

In [ ]:
import pandas as pd

df = geemap.ee_to_df(zonales_full)

# Nombres más manejables que los que vienen por defecto
df = df.rename(columns={'nombre': 'municipio'})

df = df.sort_values('elevation_mean', ascending=False).reset_index(drop=True)
df.round(1)

Y ya estamos en terreno conocido: ordenar, filtrar, derivar columnas, cruzar con un Excel
propio. Nada de esto se puede hacer en el Code Editor sin exportar antes.

In [ ]:
# Una columna derivada: cuánto varía la elevación dentro del municipio
df['rango_elev'] = df['elevation_max'] - df['elevation_min']

# ¿Qué municipio tiene la pendiente media más alta?
print(df.loc[df['slope_mean'].idxmax(), 'municipio'])

df[['municipio', 'elevation_mean', 'rango_elev', 'slope_mean', 'elevation_count']].round(1)

## 7. Una figura

Cinco municipios y **una** magnitud por municipio: eso es una barra horizontal ordenada de
mayor a menor. Horizontal porque las etiquetas son nombres y así se leen sin girar la
cabeza; ordenada porque el orden alfabético no dice nada y el de magnitud es la mitad del
mensaje.

Tres decisiones que no son de gusto:

- **Un solo color.** Hay una sola serie, así que el color no codifica nada: pintar cada
  barra de un color distinto inventa una identidad que no existe y deja la figura
  ilegible para quien no distingue esos tonos.
- **Sin leyenda.** Con una serie, el título ya dice qué se está mirando.
- **El valor en la punta de cada barra**, y entonces el eje x sobra. Una cosa o la otra,
  no las dos.

In [ ]:
import matplotlib.pyplot as plt

d = df.sort_values('elevation_mean')          # ascendente: la mayor queda arriba
AZUL = '#3a6ea5'                              # una serie, un color
TINTA = '#2b2b2b'

fig, ax = plt.subplots(figsize=(7, 3.2), dpi=140)

ax.barh(d['municipio'], d['elevation_mean'], color=AZUL, height=0.6)

# El valor, directo en la punta de cada barra
for y, v in zip(d['municipio'], d['elevation_mean']):
    ax.text(v + 15, y, f'{v:,.0f} m'.replace(',', '.'),
            va='center', ha='left', fontsize=9, color=TINTA)

ax.set_title('Elevación media por municipio (SRTM, 90 m)',
             fontsize=11, color=TINTA, loc='left', pad=12)

# Quitar todo lo que no es dato: sin eje x (ya están las etiquetas), sin marco
ax.set_xlim(0, d['elevation_mean'].max() * 1.22)
ax.get_xaxis().set_visible(False)
for lado in ('top', 'right', 'bottom', 'left'):
    ax.spines[lado].set_visible(False)
ax.tick_params(axis='y', length=0, labelsize=10, colors=TINTA)

plt.tight_layout()
plt.show()

## 8. Sacar el dato fuera

Tres salidas, por orden de cuándo usar cada una:

| Salida | Cuándo |
|---|---|
| `df.to_csv` | la tabla ya está en pandas y es pequeña — lo normal |
| `Export.table.toDrive` | la tabla es grande o el cálculo tarda: va a la cola de tareas |
| `toAsset` | el resultado lo va a usar otro script de Earth Engine |

La primera es local e inmediata; las otras dos son tareas del servidor y hay que ir a
mirarlas (`ee.batch.Task.list()` o la pestaña *Tasks*).

In [ ]:
df.to_csv('estadisticas_municipios.csv', index=False)
print('guardado: estadisticas_municipios.csv')

# En Colab, para bajarlo al ordenador:
# from google.colab import files; files.download('estadisticas_municipios.csv')

In [ ]:
tarea = ee.batch.Export.table.toDrive(
    collection=zonales_full,
    description='zonales_municipios_2026',
    folder='GEE_Curso_2026',
    fileFormat='CSV',
)
tarea.start()
print('tarea lanzada:', tarea.status()['state'])

## 9. Los tres errores que todo el mundo se come

Tarde o temprano salen los tres. Lo que importa no es evitarlos, es **leer cuál es** y
saber qué palanca tocar, porque las palancas no son intercambiables.

| Error | Qué está pasando | Qué tocar |
|---|---|---|
| `User memory limit exceeded` | un trozo del cálculo no cabe en la memoria de un nodo | **`tileScale`** (2, 4, 8, 16): parte el trabajo en teselas más pequeñas. Cuesta tiempo, no dinero |
| `Computation timed out` | el cálculo entero no cabe en los 5 minutos de la petición interactiva | **exportar** en vez de imprimir: una tarea no tiene ese límite. O subir `scale` |
| `Too many concurrent aggregations` | demasiadas reducciones a la vez, casi siempre un bucle de `reduceRegion` | **`reduceRegions`** en lugar del bucle |

Y las dos que se confunden entre sí:

- **`maxPixels`** es un **seguro**, no un acelerador. Subirlo no hace nada más rápido:
  solo autoriza a leer más píxeles. Si lo tienes que subir mucho, la pregunta buena es si
  la `scale` que pediste es la que necesitas.
- **`bestEffort=True`** no arregla nada, **cambia la pregunta**: si no cabe, Earth Engine
  sube la `scale` por su cuenta hasta que quepa y te devuelve un número a una resolución
  que no elegiste y que no te dice. Está bien para explorar; en un resultado que vas a
  publicar, nunca.

In [ ]:
# Lo de arriba, aplicado: los 785 municipios de Andalucía de una vez.
# Sin tileScale esto suele dar 'User memory limit exceeded'.
zonales_todos = full.select(['elevation', 'slope']).reduceRegions(
    collection=andalucia.select(['nombre']),
    reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
    scale=90,
    tileScale=4,          # <-- la palanca de la memoria
)

# Y se exporta en vez de imprimirse, que es la palanca del tiempo
tarea_todos = ee.batch.Export.table.toDrive(
    collection=zonales_todos,
    description='zonales_andalucia_completa',
    folder='GEE_Curso_2026',
    fileFormat='CSV',
)
tarea_todos.start()
print('tarea lanzada:', tarea_todos.status()['state'])

## 10. Lo que hay que recordar

1. Más de una geometría ⇒ **`reduceRegions`**. Un bucle de `reduceRegion` es el camino
   directo a `Too many concurrent aggregations`.
2. **`scale` siempre explícito.** Es parte del resultado, no un ajuste de calidad.
3. **Reductores combinados**: una pasada, todas las estadísticas. Y llévate el `count`.
4. **`.getInfo()` y `ee_to_df` son la frontera.** Antes no ha corrido nada; ahí corre
   todo. Que estén solas en su celda.
5. `tileScale` para la memoria, exportar para el tiempo, `reduceRegions` para la
   concurrencia. Cada error tiene su palanca.
6. `bestEffort` no arregla, cambia la pregunta en silencio.

## 11. Ejercicio

1. Cambia los cinco municipios por los de **tu** provincia y saca la tabla completa de
   estadísticas a CSV.
2. Añade una columna con el **rango de elevación** y ordena por ella. ¿El municipio más
   alto es el más accidentado? ¿Por qué no tienen que coincidir?
3. Repite el cálculo a `scale=90` y a `scale=500` y compara las medias. ¿Cuánto se
   mueven? ¿Hay alguna que se mueva mucho más que las otras, y qué tienen de particular
   esos municipios?
4. Cruza la tabla con un CSV tuyo (población, superficie, lo que tengas a mano) con
   `pd.merge` y saca una relación. Esto es exactamente lo que no se puede hacer sin salir
   del Code Editor.
5. Avanzado: monta la misma tabla sobre la **serie mensual** de `series_rejilla_temporal.js`
   — una fila por municipio y mes — y mira si las barras de arriba cambian de orden según
   la época del año.